# Word Embeddings Demonstration In Python

## Background

In the real world, large amounts of data come in the form of text—such as reviews, feedback, and social media posts. However, machines cannot understand text directly.

Traditional methods like Bag of Words (BoW) and TF-IDF convert text into numbers but fail to capture meaning and relationships between words. For example, they treat “good” and “excellent” as unrelated and struggle with phrases like “not good”.

To address this, Word Embeddings were introduced, where words are represented as vectors that capture their semantic meaning and relationships, enabling machines to better understand language.


## Objective

To build a sentiment classification model that predicts whether a movie review is positive or negative using Word2Vec embeddings and Logistic Regression.

## Dataset Description

We use the IMDB Movie Reviews Dataset.
- Source: Hugging Face (load_dataset("imdb"))
- Domain: Movie reviews
- Task: Binary sentiment classification


| Feature | Description                            |
| ------- | -------------------------------------- |
| `text`  | Movie review (raw text)                |
| `label` | Sentiment (0 = negative, 1 = positive) |


| Split | Records |
| ----- | ------- |
| Train | 25,000  |
| Test  | 25,000  |


### Import Libraries

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
nltk.download('punkt')

import warnings
warnings.filterwarnings('ignore')

### Load Dataset

In [ ]:
dataset = load_dataset("imdb")

train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

In [3]:
train_df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [4]:
test_df.head()

,text,label
0,I love sci-fi and am willing to put up with a ...,0
1,"Worth the entertainment value of a rental, esp...",0
2,its a totally average film with a few semi-alr...,0
3,STAR RATING: ***** Saturday Night **** Friday ...,0
4,"First off let me say, If you haven't enjoyed a...",0


### Data Preprocessing

In [5]:
def preprocess(text):
    text = text.lower() #convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text) #Remove specical characters
    tokens = word_tokenize(text) #word tokenization
    return tokens

# Apply preprocessing
train_tokens = train_df['text'].apply(preprocess)
test_tokens = test_df['text'].apply(preprocess)

y_train = train_df['label']
y_test = test_df['label']

In [6]:
train_tokens.head() #post data cleaning

0    [i, rented, i, am, curiousyellow, from, my, vi...
1    [i, am, curious, yellow, is, a, risible, and, ...
2    [if, only, to, avoid, making, this, type, of, ...
3    [this, film, was, probably, inspired, by, goda...
4    [oh, brotherafter, hearing, about, this, ridic...
Name: text, dtype: object

In [7]:
y_train.value_counts() #y_train is labeled data 0 = negative review and 1 = positive review

label
0    12500
1    12500
Name: count, dtype: int64

### Train WORD2VEC Model (on train data only)


Word2Vec is a neural network-based model that converts words into dense vectors such that:

Words with similar meanings have similar vector representations

Example:
- “good” ≈ “excellent”

- “bad” ≈ “terrible”

It learns this using:

- Context of surrounding words
- Large corpus of text

In [8]:
# Initialize Word2Vec model
w2v_model = Word2Vec(
    sentences=train_tokens,  # tokenized sentences
    vector_size=100,         # embedding size
    window=5,                # context window
    min_count=2,             # ignore rare words
    workers=4                # CPU cores
)

### Get Sentence into Vector Form

Word2Vec gives word-level embeddings, but our problem is:

**Sentence / review classification**

So we:

- Convert each review → list of word vectors
- Take average of vectors → sentence embedding

In [9]:
def get_sentence_vector(tokens, model):
    """
    This function converts a list of tokens (words) into a single sentence vector
    using a trained Word2Vec model.
    """
    
    vectors = []  # This will store word vectors for each token
    
    # Loop through each word in the tokenized sentence
    for word in tokens:
        
        # Check if the word exists in the Word2Vec vocabulary
        if word in model.wv:
            
            # If yes, get its vector representation and store it
            vectors.append(model.wv[word])
    
    # If no words from the sentence were found in the vocabulary
    if len(vectors) == 0:
        
        # Return a zero vector of same size as Word2Vec vectors
        return np.zeros(model.vector_size)
    
    # Otherwise, take the average of all word vectors
    # This gives a single fixed-size vector representing the sentence
    return np.mean(vectors, axis=0)


# Convert all training sentences into vectors
# Each sentence → one vector
X_train = np.array([
    get_sentence_vector(tokens, w2v_model)  # apply function on each sentence
    for tokens in train_tokens              # loop through all tokenized training sentences
])


# Convert all test sentences into vectors
X_test = np.array([
    get_sentence_vector(tokens, w2v_model)  # same process for test data
    for tokens in test_tokens
])

In [10]:
X_train

array([[-0.22217724,  0.0660665 ,  0.18870778, ..., -0.18264107,
         0.12151445,  0.25167635],
       [-0.23724799,  0.10586238, -0.13345134, ..., -0.3163525 ,
         0.07454289,  0.16762766],
       [-0.13259712, -0.0108259 , -0.03984275, ..., -0.09681575,
        -0.12065474,  0.68791217],
       ...,
       [ 0.0229754 ,  0.22505234,  0.18300118, ..., -0.21872267,
        -0.12566689,  0.54685116],
       [-0.23166974,  0.1406339 ,  0.04489543, ..., -0.47049809,
        -0.06012921,  0.20210321],
       [-0.32357812,  0.13974054,  0.13713029, ..., -0.2985261 ,
         0.11073576,  0.26046106]], shape=(25000, 100), dtype=float32)

**X_train is a matrix of 25,000 sentences where each sentence is represented as a 100-dimensional vector obtained by averaging its Word2Vec word embeddings.**


**25,000** → number of sentences (data points) in your training set

**100** → number of features (dimensions) in each sentence vector (embedding size)

Because ML models need fixed-size inputs, we convert every sentence—no matter its length—into a 100-dimensional vector

### Train Logistic Regression

In [11]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### Get Predictions and Evaluate

In [12]:
# PREDICTIONS
y_pred = clf.predict(X_test)

# EVALUATION
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.80232

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.80      0.80     12500
           1       0.80      0.80      0.80     12500

    accuracy                           0.80     25000
   macro avg       0.80      0.80      0.80     25000
weighted avg       0.80      0.80      0.80     25000

